In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "realtime_fraud_detection.silver.cleaned_transactions"
GOLD_HOURLY  = "realtime_fraud_detection.`03_gold`.hourly_fraud_summary"
GOLD_CLASS   = "realtime_fraud_detection.`03_gold`.fraud_class_summary"

spark.sql("CREATE SCHEMA IF NOT EXISTS realtime_fraud_detection.`03_gold`")

silver_df = spark.read.table(SILVER_TABLE)
silver_count = silver_df.count()

# --- Gold table 1: hourly fraud trend ---
# Time = seconds since the first transaction, so hours are relative to dataset start
hourly_df = (
    silver_df
    .withColumn("hour_since_start", F.floor(F.col("Time") / 3600).cast("int"))
    .groupBy("hour_since_start")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.col("Class").cast("long")).alias("fraud_transactions"),
        F.sum("Amount").alias("total_amount"),
        F.sum(F.when(F.col("Class") == 1, F.col("Amount")).otherwise(0)).alias("fraud_amount"),
        F.avg("Amount").alias("avg_amount"),
        F.max("Amount").alias("max_amount"),
    )
    .withColumn("fraud_rate_pct",
                F.round(F.col("fraud_transactions") / F.col("total_transactions") * 100, 4))
    .withColumn("hour_of_day_rel", F.col("hour_since_start") % 24)
    .orderBy("hour_since_start")
)

# --- Gold table 2: fraud vs legitimate profile ---
class_df = (
    silver_df
    .groupBy("Class")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("Amount").alias("total_amount"),
        F.avg("Amount").alias("avg_amount"),
        F.stddev("Amount").alias("stddev_amount"),
        F.min("Amount").alias("min_amount"),
        F.max("Amount").alias("max_amount"),
        F.percentile_approx("Amount", 0.5).alias("median_amount"),
    )
    .withColumn("label", F.when(F.col("Class") == 1, "fraud").otherwise("legit"))
    .withColumn("pct_of_transactions",
                F.round(F.col("total_transactions") / F.lit(silver_count) * 100, 4))
)

# --- Reconciliation: Gold must match Silver ---
assert hourly_df.agg(F.sum("total_transactions")).collect()[0][0] == silver_count, "Hourly Gold != Silver"
assert class_df.agg(F.sum("total_transactions")).collect()[0][0] == silver_count, "Class Gold != Silver"

# --- Write (overwrite = safe to rerun) ---
for df, table in [(hourly_df, GOLD_HOURLY), (class_df, GOLD_CLASS)]:
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table))

print(f"🥇 Gold tables written from {silver_count} Silver rows.")
display(class_df)
display(hourly_df)